In [458]:
import pandas as pd
import plotly.express as px
from scipy.stats import describe
import plotly.graph_objects as go
from matplotlib import pyplot as plt

In [459]:
# CPIAUCSL
# CPILFESL
# WPSFD49207
# WPSFD4131
# PCEPILFE
# PCEPI

INDICATOR = 'PCEPI'

MoM_col = 'MoM ^ %'
YoY_col = 'YoY ^ %'

# COL = MoM_col
COL = YoY_col

In [460]:
df = pd.read_csv('../../data/' + INDICATOR + '.csv')

df['DATE'] = pd.to_datetime(df['DATE'])
df.set_index('DATE', inplace=True)

# df[MoM_col] = df[INDICATOR].pct_change(fill_method=None) * 100
df[MoM_col] = df[INDICATOR].ffill().pct_change() * 100

# df[YoY_col] = df[INDICATOR].pct_change(fill_method=None, periods=12) * 100
df[YoY_col] = df[INDICATOR].ffill().pct_change(periods=12) * 100

df.tail()

,PCEPI,MoM ^ %,YoY ^ %
DATE,,,
2025-05-01,126.380,0.182323,2.458086
2025-06-01,126.743,0.287229,2.593513
2025-07-01,126.957,0.168846,2.603123
2025-08-01,127.287,0.259931,2.742778
2025-09-01,127.626,0.266327,2.788248


- **Overview on change**

In [461]:
positive_filter = df[COL] > 0
negative_filter = df[COL] < 0
zero_filter = df[COL] == 0

# average change
avg_change = []

# frequency
frequency = []

# frequency pct
pct = []

# Prob adj
prob_adj = []

p_count = df[positive_filter][COL].count()
n_count = df[negative_filter][COL].count()
z_count = df[zero_filter][COL].count()

if p_count > 0:
    avg_change.append(df[positive_filter][COL].mean())
    frequency.append(p_count)
    pct.append(100 * p_count / len(df))
    prob_adj.append(pct[0] * avg_change[0])
else:
    avg_change.append(0.0)
    frequency.append(0)
    pct.append(0.0)
    prob_adj.append(0.0)

if n_count > 0:
    avg_change.append(df[negative_filter][COL].mean())
    frequency.append(n_count)
    pct.append(100 * n_count / len(df))
    prob_adj.append(pct[1] * abs(avg_change[1]))
else:
    avg_change.append(0.0)
    frequency.append(0)
    pct.append(0.0)
    prob_adj.append(0.0)

if z_count > 0:
    avg_change.append(0.0)
    frequency.append(z_count)
    pct.append(100 * z_count / len(df))
    prob_adj.append(0.0)
else:
    avg_change.append(0.0)
    frequency.append(0)
    pct.append(0.0)
    prob_adj.append(0.0)

avg_change.append('')
pct.append('')
prob_adj.append('')
if n_count > 0: frequency.append(frequency[0]/frequency[1])
else: frequency.append('')

pd.DataFrame({
    "MoM ^ %": [x for x in avg_change],
    "Freq": [f"{x}" for x in frequency],
    "Freq %": pct,
    "Prob Adjust": prob_adj,
}, index=["Av Pos", "Av Neg", "Zero", "Ratio P/N"])

,MoM ^ %,Freq,Freq %,Prob Adjust
Av Pos,3.334135,780,97.378277,324.672367
Av Neg,-0.759155,9,1.123596,0.852983
Zero,0.0,0,0.0,0.0
Ratio P/N,,86.66666666666667,,


In [462]:
# positive_filter = df[COL] > 0
# negative_filter = df[COL] < 0
# zero_filter = df[COL] == 0

# # count
# total_items = len(df)

# # average change
# avg_change = [
#     df[positive_filter][COL].mean(),
#     df[negative_filter][COL].mean(),
#     0,
#     ""
# ]

# # frequency
# frequency = [
#     df[positive_filter][COL].count(),
#     df[negative_filter][COL].count(),
#     df[zero_filter][COL].count()
# ]
# frequency.append(frequency[0]/frequency[1])

# # frequency pct
# pct = [100 * frequency[i]/total_items for i in range(3)]
# pct.append("")

# # Prob adj
# prob_adj = [pct[i] * avg_change[i] for i in range(3)]
# prob_adj.append("")

# pd.DataFrame({
#     "MoM ^ %": [x for x in avg_change],
#     "Freq": [f"{x}" for x in frequency],
#     "Freq %": pct,
#     "Prob Adjust": prob_adj,
# }, index=["Av Pos", "Av Neg", "Zero", "Ratio P/N"])

- **Stats**

In [463]:
stats = describe(df[COL].dropna().tolist())

obj = {
   'nobs': str(stats.nobs),
   'Min %': stats.minmax[0],
   'Max %': stats.minmax[1],
   'Mean %': stats.mean,
   'Median %': df[COL].median(),
   'Mode %': df[COL].mode(dropna=True)[0],
   'Variance': stats.variance,
   'Skewness': stats.skewness,
   'Kurtosis': stats.kurtosis
}

df_stats = (
    pd.DataFrame(list(obj.items()), columns=["Metric", "Value"])
      .set_index("Metric")
)
df_stats



,Value
Metric,
nobs,789
Min %,-1.466009
Max %,11.595182
Mean %,3.287444
Median %,2.568966
Mode %,-1.466009
Variance,5.751641
Skewness,1.418893
Kurtosis,1.891629


- **Data preview**

In [464]:
# Define bins

N = 6

alpha = (stats.minmax[1] - stats.minmax[0]) / N

bins = [stats.minmax[0] + i * alpha for i in range(N + 1)]

bin_labels = [f"{round(bins[i], 2)}% to {round(bins[i+1], 2)}%" for i in range(len(bins) - 1)]
bin_labels[0] = f"Less than {round(bins[1],2)}%"
bin_labels[-1] = f"Greater than {round(bins[-2], 2)}%"

# Assign data to bins
binned = pd.cut(df[COL], bins=bins, labels=bin_labels, include_lowest=True)

# Calculate frequency, probability, and cumulative probability
frequency = binned.value_counts().sort_index()
probability = 100 * frequency / frequency.sum()
cumulative_probability = probability.cumsum()

occurrence_frequencies = pd.DataFrame({
    'Frequency': frequency.values,
    'Probability %': probability.values,
    'Cumulative Probability %': cumulative_probability.values
}, index=bin_labels)

occurrence_frequencies

,Frequency,Probability %,Cumulative Probability %
Less than 0.71%,35,4.435995,4.435995
0.71% to 2.89%,416,52.724968,57.160963
2.89% to 5.06%,207,26.235741,83.396705
5.06% to 7.24%,70,8.871990,92.268695
7.24% to 9.42%,28,3.548796,95.817490
Greater than 9.42%,33,4.182510,100.000000


In [465]:
fig = px.bar(
    occurrence_frequencies,
    x=occurrence_frequencies.index,
    y='Probability %',
)

fig.update_layout(
    title=dict(text='CPIAUCSL % change'),
    width=800,
    height=600,
    yaxis=dict(title="Probability %"),
    hovermode="x unified",
    template="plotly_dark"
)

fig.show()

In [466]:
# COL = YoY_col

In [467]:
fig = px.bar(df, x=df.index, y=COL)

fig.update_layout(
    title=dict(text=INDICATOR + ' ' + COL),
    width=1300,
    height=600,
    yaxis=dict(title="%"),
    hovermode="x unified",
    template="plotly_dark"
)

# Reference line
fig.add_trace(go.Scatter(
    x=df.index,
    y=[df[COL].mean()] * len(df),
    name='Average ' + COL,
    line=dict(
        width=2,
        color="#F2C14E",   # muted gold
        dash="dash"
    ),
    # hoverinfo="skip",
    # showlegend=False,
))

fig.show()